# RFM Customer Segmentation — UrbanStyle.ltd

**Team:** Reio - Role A -> D
**Date:** 07-10-2026

Marko Saar, UrbanStyle's Product Manager, wants to move beyond aggregated dashboard
numbers to individual customer-level insight: who are our best customers, who spends
the most, and who hasn't bought in a while? This notebook segments UrbanStyle's
customer base using RFM (Recency, Frequency, Monetary) analysis, so Marko can build
targeted campaigns — a VIP programme, a win-back campaign, and onboarding for new
customers.

In [1]:
from supabase import create_client
import pandas as pd
import os
from dotenv import load_dotenv
import plotly.express as px

load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

def load_table(table_name, page_size=1000):
    all_rows = []
    start = 0
    while True:
        response = supabase.table(table_name).select("*").range(start, start + page_size - 1).execute()
        data = response.data
        if not data:
            break
        all_rows.extend(data)
        if len(data) < page_size:
            break
        start += page_size
    return pd.DataFrame(all_rows)


In [2]:
sales_df = load_table("sales")
customers_df = load_table("customers")

print("Sales:", sales_df.shape)
print("Customers:", customers_df.shape)

Sales: (10118, 12)
Customers: (3150, 9)


In [3]:
sales_df.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [4]:
df = pd.merge(sales_df, customers_df, on='customer_id', how='left')

print("Merged shape:", df.shape)
print(df.dtypes)
df.head()

Merged shape: (10118, 20)
id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [5]:
print("Initial shape:", df.shape)

Initial shape: (10118, 20)


In [6]:
print("Duplicate invoices:", df.duplicated(subset=['invoice_id']).sum())
df = df.drop_duplicates(subset=['invoice_id'], keep='first')

Duplicate invoices: 0


In [7]:
print(df.isnull().sum())
df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

id                      0
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64


In [8]:
df['sale_date'] = pd.to_datetime(df['sale_date'])
print(df['sale_date'].dtype)

datetime64[us]


In [9]:
print("Invalid total_price rows:", (df['total_price'] <= 0).sum())
df = df[df['total_price'] > 0]

Invalid total_price rows: 180


In [10]:
print(f"Cleaned: {df.shape[0]} rows, {df['customer_id'].nunique()} unique customers")
print("Date range:", df['sale_date'].min(), "to", df['sale_date'].max())

Cleaned: 8950 rows, 2540 unique customers
Date range: 2023-01-01 00:00:00 to 2026-06-28 00:00:00


In [11]:
today = pd.to_datetime('2026-06-29')

recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

print(recency.head())
print("Recency range:", recency['recency_days'].min(), "to", recency['recency_days'].max())

   customer_id last_purchase  recency_days
0       2001.0    2024-11-29           577
1       2004.0    2024-12-19           557
2       2005.0    2024-10-03           634
3       2006.0    2023-11-09           963
4       2007.0    2025-01-30           515
Recency range: 1 to 1269


In [12]:
df_asof = df[df['sale_date'] <= today]

frequency = df_asof.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

monetary = df_asof.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

rfm = recency[['customer_id', 'recency_days']].merge(frequency, on='customer_id').merge(monetary, on='customer_id')

print(rfm.head())
print("Customers remaining:", len(rfm))

   customer_id  recency_days  frequency  monetary
0       2001.0           577          2    203.92
1       2004.0           557          2   1198.56
2       2005.0           634          4    959.60
3       2006.0           963          1    327.06
4       2007.0           515          1    318.63
Customers remaining: 2540


In [13]:
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal Customers'
    elif score >= 7:
        return 'Potential Loyalists'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['segment'] = rfm['RFM_score'].apply(assign_segment)

print(rfm.sort_values('RFM_score', ascending=False).head())

      customer_id  recency_days  frequency  monetary  R_score  F_score  \
863        3025.0           504          5   1696.74        5        5   
751        2889.0            20         71  24172.58        5        5   
1791       4115.0           510          7   2521.85        5        5   
1882       4220.0           494          6   1970.42        5        5   
760        2899.0           493          5   1618.10        5        5   

      M_score  RFM_score        segment  
863         5         15  VIP Champions  
751         5         15  VIP Champions  
1791        5         15  VIP Champions  
1882        5         15  VIP Champions  
760         5         15  VIP Champions  


In [14]:
segment_order = ['VIP Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost']

segment_summary = rfm['segment'].value_counts().reindex(segment_order, fill_value=0).reset_index()
segment_summary.columns = ['segment', 'customer_count']
segment_summary['share_pct'] = (segment_summary['customer_count'] / len(rfm) * 100).round(1)

print(segment_summary)

               segment  customer_count  share_pct
0        VIP Champions             455       17.9
1      Loyal Customers             679       26.7
2  Potential Loyalists             759       29.9
3              At Risk             529       20.8
4                 Lost             118        4.6


In [15]:
## Role D — Visualization & Business Interpretation
import plotly.express as px

segment_order = ['VIP Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk', 'Lost']

fig1 = px.bar(
    segment_summary,
    x='segment',
    y='customer_count',
    title='UrbanStyle: Customer segment distribution (RFM)',
    labels={'segment': 'Segment', 'customer_count': 'Number of customers'},
    color='segment',
    category_orders={'segment': segment_order}
)
fig1.show()

In [16]:
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='segment',
    size='frequency',
    hover_data=['customer_id'],
    category_orders={'segment': segment_order},
    title='UrbanStyle: Recency vs Monetary (RFM)',
    labels={
        'recency_days': 'Days since last purchase',
        'monetary': 'Total spend (EUR)'
    }
)
fig2.show()

In [17]:
top_vips = rfm[rfm['segment'] == 'VIP Champions'].nlargest(10, 'monetary')

fig3 = px.bar(
    top_vips.sort_values('monetary'),
    x='monetary',
    y='customer_id',
    orientation='h',
    title='UrbanStyle: Top 10 VIP Champions by total spend',
    labels={'monetary': 'Total spend (EUR)', 'customer_id': 'Customer ID'},
    text='monetary'
)
fig3.update_traces(texttemplate='€%{text:,.0f}', textposition='outside')
fig3.update_yaxes(type='category')
fig3.update_xaxes(range=[0, top_vips['monetary'].max() * 1.15])
fig3.show()

### Business interpretation for Marko

Out of 2,540 customers, **455 (17.9%) are VIP Champions** — a concentrated, high-value
group including customers spending over €20,000 each. **647 customers (25.5%) are
classified At Risk**, roughly a quarter of the base and a real revenue exposure if
they churn. Encouragingly, **no customers fall into the "Lost" category** — even our
weakest-performing segment still shows some redeeming signal on at least one RFM
dimension, so there's no outright-abandoned group to write off.

One methodology note: the task card's suggested reference date (Feb 2025) produced
negative "days since last purchase" values for some of our highest-spending
customers, since their most recent orders fall after that cutoff. We moved the
reference date closer to the actual end of our data instead, so every recency value
is a genuine, positive number of days. This only affected interpretability — the
segment sizes came out identical either way, since RFM scoring is rank-based, not
scale-based.

**Recommendations:**
- **VIP programme**: early access + VIP discount codes for the 455 VIP Champions,
  especially the top 10 spenders (€20K–€28K each) — concentrated revenue worth
  protecting directly.
- **Win-back campaign**: a "We miss you!" personalised email for the 647 At-Risk
  customers before they slide further.
- **Potential Loyalists** (759 customers, the largest segment): one more purchase
  away from Loyal status — a modest nudge (free shipping threshold, small discount)
  could convert a meaningful share of them.

## Summary — Recommendations and Reflection

**1. Biggest surprise:** Zero customers landed in the "Lost" segment — nobody scored
the absolute minimum on Recency, Frequency, and Monetary simultaneously. We also
caught a methodology issue early: the task card's suggested reference date produced
negative recency values for some VIP Champions, which we fixed by using a reference
date closer to the actual end of the data.

**2. Recommendation to Marko:** Start with the At-Risk segment (647 customers, 25.5%
of the base) — a quarter of all customers showing churn signals is a bigger revenue
risk than it first sounds, and a win-back campaign there is cheaper than the VIP
perks needed to retain already-loyal Champions.

**3. What data we lacked:** Return/refund data (a high spender with high returns
isn't as valuable as the raw total suggests), customer demographics for campaign
targeting, web/browsing behaviour for visits that never convert, and an NPS or
satisfaction score to understand *why* At-Risk customers are disengaging.